# ML1 - GridSearch with DecisionTreeRegressor, RandomForestRegressor

This notebook follows the ML1_Pipeline_Model notebook and focuses on:
1. Using DecisionTreeRegressor and RandomForestRegressor (or others)  
2. Hyperparameter tuning with GridSearchCV  
3. Cross-validation

## 1. Importing Libraries

In [80]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso 
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeClassifier, export_text
from scipy.stats import uniform
from scipy.stats import randint


## 2. Loading and Preparing Data with the Pipeline

We follow the same preparation steps as in the previous notebook.


In [81]:
# Load the data from the pickle file.
df = pd.read_pickle('arbres_clean_Laura.pkl')
#Prepare the train and test sets
X = df.drop(columns=['anneedeplantation'])
y = df['anneedeplantation']

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=20)

categorical_variables = X_train.select_dtypes(
    include=['object', 'string', 'category']
).columns.tolist()

numerical_variables = X_train.select_dtypes(include = ['number']).columns.tolist()
print(categorical_variables,numerical_variables)

#pipeline
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))])

numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

preprocessor = ColumnTransformer([
    ('cat', categorical_pipeline, categorical_variables),
    ('num', numerical_pipeline, numerical_variables)])
	
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

['sous_categorie', 'adr_secteur', 'genre_bota', 'espece', 'portarbre', 'structure'] ['stadededeveloppement', 'hauteurarbre', 'latitude', 'longitude']


In [ ]:
#permet de savoir combien de variables on a après le pipeline, ici 421 colonnes
X_train_transformed.shape

(22144, 421)

## 3. Initial Cross-Validation

Before tuning hyperparameters, let's evaluate a model with default parameters using cross-validation.


In [85]:
# Initial cross-validation without hyperparameter tuning  
# We can also print the MSE or MAE scores from each cross-validation fold.

model_linear = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', LinearRegression())
])
model_lasso = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', Lasso(alpha=0.001, max_iter=10000))
])

#max_depth est plus ou moins nb de variables explorées
model_random_forest = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', RandomForestRegressor(max_depth=30, min_samples_leaf=0.05,random_state=42))
])

model_random_forest_RS = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', RandomForestRegressor(max_depth=40, min_samples_leaf=0.05,
                                       n_estimators=64, random_state=42))
])

model_decision_tree = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', DecisionTreeClassifier(max_depth=10, random_state=42))
])

model_gradient_boosting_regressor = Pipeline([
    ('preprocessor', preprocessor),
    ('regression', GradientBoostingRegressor(random_state=42))
])




In [ ]:
#application avec la régression linéaire de base
arbres_scores = cross_val_score(model_linear, X_train, y_train, cv = 5, scoring='neg_mean_squared_error') 

print("MSE moyen :", np.mean(-arbres_scores))
print("Écart-type :", np.std(-arbres_scores))

MSE moyen : 139.5175592247019
Écart-type : 6.178005661137732


## 4. 🌳 Modeling with GridSearchCV

Let's choose two machine learning models.

Evaluate the models with hyperparameter tuning (testing different hyperparameter values).  
Optimize the models as you wish, for example for Decision Tree and Random Forest:

### 🔍 Hyperparameters Tested
- `max_depth`: maximum depth of the trees [3, 4, 5, 10]  
- `min_samples_leaf`: minimum number of samples required at a leaf [0.1, 0.05]  
- `n_estimators` (for Random Forest only): number of trees in the forest [50, 100, 200]

Let's use GridSearchCV to automate the search for the best hyperparameters.

In [41]:
# Hyperparameter tuning with GridSearchCV, and we can print the MSE or MAE scores for each cross-validation fold.

param_grid = [
 {'regression__alpha': [0.001, 0.01, 0.1, 1, 3, 10, 100, 1000]}
 ]

#arbres_scores_lasso = cross_val_score(model_lasso, X_train, y_train, cv = 5, scoring='neg_mean_squared_error')
arbres_grid_search = GridSearchCV(model_lasso, param_grid, cv=5, scoring='neg_mean_squared_error')
arbres_grid_search.fit(X_train, y_train)

arbres_grid_results = pd.DataFrame(arbres_grid_search.cv_results_)

print(arbres_grid_results)
print(arbres_grid_search.best_estimator_)



   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0      13.077787      1.559408         0.025188        0.010446   
1       5.836213      0.601150         0.022996        0.005243   
2       2.109439      0.517948         0.019531        0.003653   
3       0.162483      0.022980         0.045738        0.034477   
4       0.116760      0.025880         0.018531        0.001923   
5       0.082546      0.012020         0.023179        0.003970   
6       0.066898      0.000545         0.018426        0.000874   
7       0.076387      0.007129         0.021530        0.001679   

   param_regression__alpha                        params  split0_test_score  \
0                    0.001  {'regression__alpha': 0.001}        -138.451303   
1                    0.010   {'regression__alpha': 0.01}        -144.572596   
2                    0.100    {'regression__alpha': 0.1}        -173.353017   
3                    1.000      {'regression__alpha': 1}        -221.274252   
4

In [75]:
#GridSearch avec la random forest
param_grid_forest = {
    'regression__max_depth': [400],
    'regression__min_samples_leaf': [0.05, 0.01, 0.001, 0.0001],
    'regression__n_estimators': [250, 300, 350, 400]
}

arbres_RF_grid_search = GridSearchCV(model_random_forest, param_grid_forest, cv=5, scoring='neg_mean_squared_error')
arbres_RF_grid_search.fit(X_train, y_train)

arbres_RF_grid_results = pd.DataFrame(arbres_RF_grid_search.cv_results_)

KeyboardInterrupt: 

In [43]:
print(arbres_RF_grid_results)
print(arbres_RF_grid_search.best_estimator_)

    mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0        0.645341      0.013416         0.028931        0.001230   
1        1.213884      0.006814         0.039056        0.001071   
2        2.673382      0.341543         0.062526        0.002476   
3        0.837435      0.013377         0.028416        0.000674   
4        1.595760      0.022959         0.039442        0.000758   
5        3.430156      0.137859         0.070770        0.012306   
6        0.812269      0.069100         0.031757        0.002043   
7        1.474086      0.056844         0.045016        0.007823   
8        2.815541      0.112233         0.064135        0.002377   
9        1.098131      0.118489         0.030494        0.001874   
10       1.979935      0.142282         0.041990        0.000754   
11       3.946953      0.126347         0.070238        0.003012   
12       0.835606      0.082736         0.032589        0.002581   
13       1.464827      0.105739         0.041021

In [84]:
#Plutôt que d'utiliser un gridsearch, plutôt utiliser un randomsearch, plus rapide


#RandomizedSearchCV avec la random forest

distributions = {
    'regression__max_depth': [3, 4, 5, 10, None],
    'regression__min_samples_leaf': [0.05, 0.1],
    'regression__n_estimators': randint(50, 201)
}

arbres_RF_random_search = RandomizedSearchCV(
    model_random_forest_RS,
    param_distributions=distributions,
    n_iter=10,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

arbres_RF_random_search.fit(X_train, y_train)

arbres_RF_random_results = pd.DataFrame(arbres_RF_random_search.cv_results_)

print(arbres_RF_random_results)
print(arbres_RF_random_search.best_estimator_)


   mean_fit_time  std_fit_time  mean_score_time  std_score_time  \
0       2.577013      0.035323         0.074117        0.008281   
1       1.879092      0.020088         0.087066        0.026265   
2       4.615553      0.051118         0.110851        0.005592   
3       4.620794      0.057981         0.120674        0.007965   
4       4.874317      0.053671         0.120534        0.002757   
5       3.635170      0.038563         0.100709        0.003128   
6       2.326952      0.019736         0.080739        0.003120   
7       1.912873      0.016433         0.071648        0.002199   
8       2.459869      0.020712         0.079383        0.008488   
9       3.028669      0.193310         0.060939        0.008183   

  param_regression__max_depth  param_regression__min_samples_leaf  \
0                          10                                0.05   
1                           5                                0.10   
2                           4                          

In [ ]:
#RandomizedSearchCV avec le gradient boosting

distributions = {
    'regression__max_depth': [3, 4, 5, 10, None],
    'regression__min_samples_leaf': [0.05, 0.1],
    'regression__n_estimators': randint(50, 201)
}

arbres_RF_random_search = RandomizedSearchCV(
    model_random_forest_RS,
    param_distributions=distributions,
    n_iter=10,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)

arbres_RF_random_search.fit(X_train, y_train)

arbres_RF_random_results = pd.DataFrame(arbres_RF_random_search.cv_results_)

print(arbres_RF_random_results)
print(arbres_RF_random_search.best_estimator_)


In [44]:
# Code for evaluating the best model on the test set.

# Créer un modèle de référence
dummy = DummyRegressor(strategy='mean')

# L'entraîner sur les données d'entraînement
dummy.fit(X_train_transformed, y_train)

# Prédire les années sur le test
y_pred_dummy = dummy.predict(X_test)

dummy_scores = cross_val_score(
    dummy,
    X_train_transformed,
    y_train,
    cv=10,
    scoring='neg_mean_squared_error'
)



## 5. Final Evaluation on the Test Set

In [86]:
from sklearn.model_selection import cross_validate
import numpy as np

scoring = {
    'mse': 'neg_mean_squared_error',
    'mae': 'neg_mean_absolute_error',
    'r2': 'r2'
}

# modèles Pipeline
models = {
    'LinearRegression': model_linear,
    'Lasso': model_lasso,
    'Dummy': dummy,
    'Random Forest' : model_random_forest_RS,
    'Decision Tree' : model_decision_tree,
    'Gradient Boosting' : model_gradient_boosting_regressor 
}

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=5,
        scoring=scoring
    )

    mse = -scores['test_mse']
    mae = -scores['test_mae']
    r2 = scores['test_r2']

    rmse = np.sqrt(mse)

    print(f"\n{name}")
    print(f"MSE moyen  : {mse.mean():.2f}")
    print(f"RMSE moyen : {rmse.mean():.2f}")
    print(f"MAE moyen  : {mae.mean():.2f}")
    print(f"R² moyen   : {r2.mean():.3f}")
    print(f"Écart-type MAE : {mae.std():.2f}")


LinearRegression
MSE moyen  : 139.52
RMSE moyen : 11.81
MAE moyen  : 8.92
R² moyen   : 0.575
Écart-type MAE : 0.13

Lasso
MSE moyen  : 138.95
RMSE moyen : 11.78
MAE moyen  : 8.95
R² moyen   : 0.577
Écart-type MAE : 0.13

Dummy
MSE moyen  : 328.57
RMSE moyen : 18.13
MAE moyen  : 15.90
R² moyen   : -0.000
Écart-type MAE : 0.11

Random Forest
MSE moyen  : 206.47
RMSE moyen : 14.37
MAE moyen  : 11.41
R² moyen   : 0.371
Écart-type MAE : 0.17


/usr/local/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(



Decision Tree
MSE moyen  : 275.50
RMSE moyen : 16.59
MAE moyen  : 9.30
R² moyen   : 0.161
Écart-type MAE : 0.27

Gradient Boosting
MSE moyen  : 133.48
RMSE moyen : 11.55
MAE moyen  : 9.02
R² moyen   : 0.594
Écart-type MAE : 0.09


Exception ignored in: <function ResourceTracker.__del__ at 0x10d1de520>
Traceback (most recent call last):
  File "/usr/local/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/usr/local/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/usr/local/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x10665e520>
Traceback (most recent call last):
  File "/usr/local/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/usr/local/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/usr/local/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x102c3e5

## 6. Visualization of Predictions  
One model will suffice.

In [47]:
# Visualisation of predictions.

## [OPTIONAL] 7.Feature Importance

Using the random forest, how can we determine which features are the most important?

In [48]:
# Code. In your opinion, how is feature importance calculated in Random Forest?

Typically, one might consider using backward elimination on feature importance to select the best subset of features, but this can be time-consuming if there are many features and the model takes a while to train, as is the case here. Instead, we can use Lasso-regularized regression to eliminate features, before retraining a RandomForest on the selected features. Create a pipeline with `LassoCV` as the selector and a RandomForest as the model with the best hyperparameters found by the previous grid-search. Evaluate on the test set.

In [49]:
# Code

## 8. [OPTIONAL]  Compare more different models

Compare more regression models with one another (for example, in addition to linear regression, decision trees, and forests, use `GradientBoostingRegressor`, and the novel `TabICLRegressor` if you have time ( https://github.com/soda-inria/tabicl ))), by performing appropriate hyperparameter tuning for each model:

In [50]:
# Code. Search which hyper-parameters are worth tuning for each model

Choose the model that performs best and its optimal hyperparameters, then select the best subset of features for this model, and finally perform one last grid search on your model’s hyperparameters using the selected features to obtain your final model. Evaluate it on the test set.

In [51]:
# Code

## Conclusion